author: ransell

# Bradley-Terry Analysis


**Note:** Ensure that any dataset paths are set at the top of this notebook before running all cells.

**Bradley-Terry Rankings for X and YT pairs**

At this point -> the more positive the normalized ranking between -1 and 1, the more AI like the post. 

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

def run_bt_analysis(file_path, output_csv):
    data = pd.read_csv(file_path)
    data = data[data['post_winner'] != data['post_loser']].reset_index(drop=True)
    
    pairs = pd.DataFrame({'winner': data['post_winner'], 'loser': data['post_loser']})
    winners = pairs['winner'].dropna().astype(str).unique()
    losers  = pairs['loser'].dropna().astype(str).unique()
    teams = pd.Series(np.union1d(winners, losers))

    pairs['winner_code'] = pd.Categorical(pairs['winner'].astype(str), categories=teams).codes
    pairs['loser_code']  = pd.Categorical(pairs['loser'].astype(str), categories=teams).codes

    X = np.zeros((len(pairs), len(teams)))
    X[np.arange(len(pairs)), pairs['winner_code']] = 1
    X[np.arange(len(pairs)), pairs['loser_code']]  = -1
    y = np.ones(len(pairs))

    model = sm.GLM(y, X[:, 1:], family=sm.families.Binomial())
    results = model.fit()

    abilities = np.concatenate(([0], results.params))
    norm = -1 + 2 * (np.argsort(np.argsort(abilities)) / (len(abilities) - 1))

    rankings = []
    for i, post in enumerate(teams):
        if post in data['post_winner'].astype(str).values:
            text_id = data[data['post_winner'].astype(str) == post].iloc[0]['id_winner']
        else:
            text_id = data[data['post_loser'].astype(str) == post].iloc[0]['id_loser']
        rankings.append({'Text': post, 'Text_ID': int(text_id), 'Ability': abilities[i], 'Normalized Ability': norm[i]})
    
    pd.DataFrame(rankings).to_csv(output_csv, index=False)
    print(f"Saved: {output_csv}")

# Run for both platforms
run_bt_analysis("data/tt_data.csv",  "data/bt_tt_posts_rankings.csv")
run_bt_analysis("data/ts_data.csv", "data/bt_ts_posts_rankings.csv")


Saved: data/bt_tt_posts_rankings.csv
Saved: data/bt_ts_posts_rankings.csv


Merging in metadata from selected_posts

In [2]:
import pandas as pd
import numpy as np

def merge_bt_with_model_and_persona(
    bt_path,           
    posts_path,        # Path to model + human post metadata CSV
    post_text_col,     
    output_path,       
    label             
):
    print(f"\n--- Merging for {label} ---")

    # Load data
    bt_df = pd.read_csv(bt_path)
    posts_df = pd.read_csv(posts_path)

    # Normalize text
    bt_df['Text'] = bt_df['Text'].astype(str).str.strip()
    posts_df[post_text_col] = posts_df[post_text_col].astype(str).str.strip()

    # Separate AI and human posts
    ai_df = posts_df[posts_df['source'].str.lower() != 'human']
    human_df = posts_df[posts_df['source'].str.lower() == 'human']

    # Drop duplicate posts to prevent merge explosion
    ai_df = ai_df.drop_duplicates(subset=post_text_col, keep='first')
    human_df = human_df.drop_duplicates(subset=post_text_col, keep='first')

    # Merge AI metadata (source, persona) by post text
    merged = pd.merge(
        bt_df,
        ai_df[[post_text_col, 'source', 'persona']],
        left_on='Text',
        right_on=post_text_col,
        how='left'
    ).drop(columns=[post_text_col], errors='ignore')

    # Derive model column: if no AI source matched, it's human
    merged['model'] = merged['source'].apply(
        lambda x: str(x).strip().lower() if pd.notna(x) else 'human'
    )

    # Set persona to NaN for human posts
    merged['persona'] = merged.apply(
        lambda row: row['persona'] if row['model'] != 'human' else np.nan,
        axis=1
    )

    # Drop duplicates just in case
    merged = merged.drop_duplicates()

    # Save
    merged.to_csv(output_path, index=False)
    print(f"Saved to {output_path} ({len(merged)} records)")

# Run for both platforms with updated names ('x' → 'tt', 'yt' → 'ts')
merge_bt_with_model_and_persona(
    bt_path='data/bt_tt_posts_rankings.csv',
    posts_path='data/tt_posts_meta.csv',
    post_text_col='generated_post',
    output_path='data/bt_tt_posts_rankings_with_source_persona.csv',
    label='TT Platform'
)

merge_bt_with_model_and_persona(
    bt_path='data/bt_ts_posts_rankings.csv',
    posts_path='data/ts_posts_meta.csv',
    post_text_col='post',
    output_path='data/bt_ts_posts_rankings_with_source_persona.csv',
    label='TS Platform'
)



--- Merging for TT Platform ---
Saved to data/bt_tt_posts_rankings_with_source_persona.csv (600 records)

--- Merging for TS Platform ---
Saved to data/bt_ts_posts_rankings_with_source_persona.csv (593 records)


**Inverting Normalized Ability, Positive = Humanlike; Negative = AI-like**

In [3]:
import pandas as pd

def invert_normalized_ability(input_path, output_path, label):
    print(f"\n--- Inverting Normalized Ability for {label} ---")
    df = pd.read_csv(input_path)

    if 'Normalized Ability' not in df.columns:
        raise ValueError("Column 'Normalized Ability' not found in input file.")

    # Invert the normalized ability score
    df['Normalized Ability'] = -df['Normalized Ability']

    # Save to a new CSV
    df.to_csv(output_path, index=False)
    print(f"Inverted scores saved to {output_path}")

# TT Platform
invert_normalized_ability(
    input_path='data/bt_tt_posts_rankings_with_source_persona.csv',
    output_path='data/bt_tt_posts_rankings_with_source_persona_inverted.csv',
    label='TT Platform'
)

# TS Platform
invert_normalized_ability(
    input_path='data/bt_ts_posts_rankings_with_source_persona.csv',
    output_path='data/bt_ts_posts_rankings_with_source_persona_inverted.csv',
    label='TS Platform'
)



--- Inverting Normalized Ability for TT Platform ---
Inverted scores saved to data/bt_tt_posts_rankings_with_source_persona_inverted.csv

--- Inverting Normalized Ability for TS Platform ---
Inverted scores saved to data/bt_ts_posts_rankings_with_source_persona_inverted.csv
